In [1]:
import gzip
import pickle
import pandas as pd
from gsp_python.gsp import GSP
from tslearn.piecewise import SymbolicAggregateApproximation as SAX

DEBUG:__init__:Creating converter from 7 to 5
DEBUG:__init__:Creating converter from 5 to 7
DEBUG:__init__:Creating converter from 7 to 5
DEBUG:__init__:Creating converter from 5 to 7


In [2]:
with gzip.open("ts_normalized.pkl.gz", "rb") as f:
    ts_normalized = pickle.load(f)

sax = SAX(n_segments=50, alphabet_size_avg=5)

# Trasformazione: l'output è un array numpy 3D: (n_ts, n_segments, n_features)
ts_sax_array = sax.fit_transform(ts_normalized)

# Riconversione in lista di DataFrame per preservare la struttura e la leggibilità
ts = []
nomi_colonne = ts_normalized[0].columns

for matrice_ts in ts_sax_array:
    df_sax = pd.DataFrame(matrice_ts, columns=nomi_colonne)
    ts.append(df_sax)

colonne_da_rimuovere = [
    'id',
    'non-wear_flag',
    'battery_voltage',
    'weekday',
    'quarter',
    'relative_date_PCIAT',
    # 'X',
    # 'Y',
    # 'Z',
    'enmo',
    'anglez',
    'light',
    'sii_binary'
]

In [3]:
gsp_dataset = []

for df in ts:
    # Eliminazione vettoriale delle colonne non necessarie
    df.drop(columns=colonne_da_rimuovere, inplace=True, errors='ignore')

    gsp_sequence = []

    for index, row in df.iterrows():
        itemset = []

        for colonna in df.columns:
            valore_sax = row[colonna]

            if pd.notna(valore_sax):
                # Concatenazione del nome colonna al valore SAX (es. 'X_2')
                item = f"{colonna}_{valore_sax}"
                itemset.append(item)

        # Inserisce l'itemset solo se contiene almeno un elemento
        if itemset:
            gsp_sequence.append(itemset)

    # Inserisce la sequenza del paziente solo se non è vuota
    if gsp_sequence:
        gsp_dataset.append(gsp_sequence)

# Verifica della struttura ottenuta
print(f"Numero totale di sequenze (pazienti): {len(gsp_dataset)}")
if len(gsp_dataset) > 0 and len(gsp_dataset[0]) > 0:
    print(f"Struttura del primo itemset: {gsp_dataset[0][0]}")

Numero totale di sequenze (pazienti): 4437
Struttura del primo itemset: ['X_2', 'Y_1', 'Z_3']


In [4]:
# gsp e run_gsp()
gsp = GSP(
    ds=gsp_dataset,
    minsup=0.6,
    mingap=0,
    maxgap=4,
    maxspan=16
)
patterns = gsp.run_gsp()

# Definizione della funzione di conversione estratta dal notebook


def output_to_dataframe(patterns):
    df = pd.DataFrame(data=patterns, columns=['pattern', 'support_count'])
    return df


# Generazione del DataFrame e ordinamento per frequenza decrescente
df_patterns = output_to_dataframe(patterns)
df_patterns = df_patterns.sort_values(
    by='support_count', ascending=False).reset_index(drop=True)

# Calcolo del numero di elementi per ogni sequenza
df_patterns['pattern_length'] = df_patterns['pattern'].apply(len)

# Identificazione della lunghezza massima nel trend
max_length = df_patterns['pattern_length'].max()

# Filtraggio del DataFrame per ottenere solo le sequenze con lunghezza massima
longest_patterns = df_patterns[df_patterns['pattern_length'] == max_length]

# Stampa a video dei risultati
print(
    f"Le sequenze con il maggior numero di elementi (lunghezza {max_length}) sono:")
print(longest_patterns[['pattern', 'support_count']])

Le sequenze con il maggior numero di elementi (lunghezza 5) sono:
                                  pattern  support_count
3229  [[Y_2], [Z_2], [Z_2], [Z_2], [Z_2]]           2932
3255  [[Z_2], [Z_2], [Z_2], [Z_3], [Z_3]]           2929
3561  [[Z_2], [Z_2], [Z_2], [Z_2], [Y_2]]           2899
3614  [[Z_3], [Z_3], [Z_2], [Z_2], [Z_2]]           2894
3624  [[Z_2], [Z_2], [Z_2], [Z_2], [Z_2]]           2893
...                                   ...            ...
6689  [[Z_2], [Z_2], [Y_3], [Z_2], [Z_2]]           2667
6691  [[Y_1], [Z_3], [Z_3], [Z_3], [Z_3]]           2667
6711  [[Z_2], [Z_3], [Z_3], [Z_2], [Y_2]]           2665
6729  [[Y_4], [Y_4], [Y_4], [Y_4], [Y_3]]           2664
6758  [[Y_1], [Y_2], [Z_3], [Z_2], [Z_2]]           2663

[120 rows x 2 columns]
